In [ ]:
!pip install -U langchain deepagents==0.6.8 langchain-chroma langchain-openai langchain-experimental langchain-community langsmith langchain-text-splitters langchain_classic google_colab


## System Documents

#### These section provides the RAG Systems Knowledge or Learnable Materials

In [ ]:
from typing import List
from langchain_core.documents import Document

documents: List = [
    Document(page_content="", metadata={"": "", "": ""}),
    Document(page_content="", metadata={"": "", "": ""}),
    Document(page_content="", metadata={"": "", "": ""}),
    Document(page_content="", metadata={"": "", "": ""}),
    Document(page_content="", metadata={"": "", "": ""}),
    Document(page_content="", metadata={"": "", "": ""}),
    Document(page_content="", metadata={"": "", "": ""}),
    Document(page_content="", metadata={"": "", "": ""}),
    Document(page_content="", metadata={"": "", "": ""}),
    Document(page_content="", metadata={"": "", "": ""}),
]


### Document Loader

In [ ]:
import tempfile


class DocumentLoader:
    def __init__(self) -> None:
        pass

    def load_text_file(self, text: str):
        with tempfile.NamedTemporaryFile(delete=False, suffix=".txt") as temp_file:
            temp_file.write(text.encode())
            temp_file_path = temp_file.name

        try:
            loader = TextLoader(temp_file_path)
            documents = loader.load()

            print(f"Loaded {len(documents)} document(s)")
            print(f"Content Preview: {documents[0].page_content[:100]}...")
            print(f"Metadata: {documents[0].metadata}...")

            for doc in documents:
                print(doc.page_content)
                print(doc.metadata)

        finally:
            os.remove(temp_file_path)

    def load_pdf_file(self, file_path: str):
        loader = PyPDFLoader(file_path)
        documents = loader.load()

        print(f"Loaded {len(documents)} document(s)")
        for i, doc in enumerate(documents):
            print(f"Doc {i+1} Content Preview: {doc.page_content[:100]}...")
            print(f"Doc {i+1} Metadata: {doc.metadata}...")


### Hybrid Retriever

#### These Section act as a Postman, Because the Document Retrives. Their words, character, paragraph like that. After Store in the Vector Space.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from typing import List


class RetrieveBuilder:
    """Combine multiple retriever using weighted reciprocal Rank Fusion"""

    def __init__(self, query, retrievers, weights, k=3, rrf_k=60) -> None:
        self.query = query
        self.retrievers = retrievers
        self.weights = weights
        self.k = k
        self.rrf_k = rrf_k

    def builder(self) -> List[Document]:
        doc_scores = {}
        for retriever, weight in zip(self.retrievers, self.weights):
            results = retriever.invoke(self.query)
            for rank, doc in enumerate(results):
                key = doc.page_content
                rrf_score = weight * (1.0 / (rank + self.rrf_k))

                if key in doc_scores:
                    doc_scores[key] = (doc_scores[key][0] + rrf_score, doc)
                else:
                    doc_scores[key] = (rrf_score, doc)

        sorted_docs = sorted(doc_scores.values(), key=lambda x: x[0], reverse=True)
        return [doc for _, doc in sorted_docs[: self.k]]


class HybridRetriever:
    """Production Hybrid Retriever with BM25 + Vector Search"""

    def __init__(
        self,
        documents: List[Document],
        bm25_weight: float = 0.5,
        k: int = 4,
    ) -> None:
        self.k = k
        self.bm25_weight = bm25_weight
        self.vector_weight = 1 - bm25_weight

        # Initialize Embeddings
        self.embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

        # Create Vector Store and Retriever
        self.vector_store = Chroma.from_documents(
            documents, self.embeddings, collection_name="hybrid_search"
        )
        self.vector_retriever = self.vector_store.as_retriever(search_kwargs={"k": k})

        # Create BM25 Retriever
        self.bm25_retriever = BM25Retriever.from_documents(documents, k=k)

    def search(self, query: str) -> List[Document]:
        """Run Hybrid Search using Weighted RRF"""
        return RetrieveBuilder(
            query=query,
            retrievers=[self.bm25_retriever, self.vector_retriever],
            weights=[self.bm25_weight, self.vector_weight],
        ).builder()

    def add_documents(self, documents: List[Document]):
        """Add New Documents to Both Retrievers"""
        # Add to Vector Store
        self.vector_store.add_documents(documents)
        # Recreate BM25(it doesn't support incremental adds)
        all_docs = self.vector_store.get()
        self.bm25_retriever = BM25Retriever.from_documents(
            [Document(page_content=doc) for doc in all_docs["documents"]], k=self.k
        )


## Token Budgeting For RAG System

In [ ]:
from typing import Dict, Tuple


class TokenBudgeting:
    def __ini__(self, max_tokens_per_request: int = 4000):
        self.max_tokens_per_request = max_tokens_per_request
        self.usage = {"total_input": 0, "total_output": 0, "request": 0}

    def estimate_tokens(self, text: str) -> int:
        """Rough Token Estimation(Actual would use TikToken)"""
        return int(len(text.split()) * 1.3)

    def check_bugs(self, text: str) -> Tuple[bool, int]:
        """Check if Request is Within Budget"""
        tokens = self.estimate_tokens(text)
        return tokens <= self.max_tokens_per_request, tokens

    def record_usage(self, input_tokens: int, output_tokens: int):
        """Record Token Usage."""
        self.usage["total_input"] += input_tokens
        self.usage["total_output"] += output_tokens
        self.usage["request"] += 1

    def get_stats(self) -> Dict:
        return {**self.usage}


### Setup Limited Token Budget LLM

> This class helps to the reduce the cost. Prevent the over payments for the APIs

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langsmith import traceable


class BudgetedLLM:
    """LLM with token budgeting"""

    def __init__(self, max_tokens: int = 4000) -> None:
        self.llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        self.budget = TokenBudgeting(max_tokens_per_request=max_tokens)

    @traceable(name="budget_invoke")
    def invoke(self, query: str) -> str:
        # Check budget
        within_budget, tokens = self.budget.check_budget(query)

        if not within_budget:
            raise ValueError(
                f"Query exceeds tokens budget: {tokens} > {self.budget.max_per_request}"
            )

        # Execute
        response = self.llm.invoke(query)
        result = response.content

        # Record Usage
        output_tokens = self.budget.estimate_tokens(result)
        self.budget.record_usage(tokens, output_tokens)

        return result

    def get_stat(self) -> Dict:
        return self.budget.get_stats()


### Simentic Chunking

> Find the most similar or equality searches for user input within the documents

In [ ]:
from langchain_text_aplitters import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbedding
from langchain_chroma import Chroma
import os
from dotenv import load_dotenv

load_dotenv()


class SemanticChunking:
    def __init__(self) -> None:
        self.embeddings = OpenAIEmbedding(model="text-embedding-3-small")
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=400,
            chunk_overlap=50,
            separator=["\n\n", "\n", ". ", " ", ""],
        )
        self.semantic_chunker = SemanticChunker(
            self.embeddings,
            breakpoint_threshold_type="percentile",
            breakpoint_threshold_amount=90,
        )

    def split(self, document: str, semantic_chunk: bool = True):
        if semantic_chunk:
            try:
                # Semantic Chunking
                semantic_chunks = self.semantic_chunker.split_text(document)
                print(f"\nSemantic Chunks: {len(semantic_chunks)}\n")
                for i, chunk in enumerate(semantic_chunks):
                    print(f"\n Chunk {i+1}: ({len(chunk)} chars)\n")
                    print(chunk[:100] + "..." if len(chunk) > 100 else chunk)
                return semantic_chunks
            except Exception as e:
                raise Exception(f"Error: {e}")
        else:
            try:
                # Traditional Chunking
                chunks = self.splitter.split_text(document)
                print(f"\nRecursive Chunks: {len(chunks)}\n")
                for i, chunk in enumerate(chunks):
                    print(f"\n Chunk {i+1}: ({len(chunk)} chars)\n")
                    print(chunk[:100] + "..." if len(chunk) > 100 else chunk)
                return chunks
            except Exception as e:
                raise Exception(f"Error: {e}")


In [ ]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_chroma import Chroma
from langchain_core.stores import InMemoryStore
from dotenv import load_dotenv

load_dotenv()


def demo_parent_document_retriever():
    """Parent Document Retriever: Small Chunks for Search, Large for Context"""

    print("=" * 60)
    print("PARENT DOCUMENT RETRIEVER")
    print("Small Chunks for Precise Search, Large Chunks for Context")
    print("=" * 60)

    long_doc = Document(
        page_content="""
  LangChain is a framework for developing applications powered by language models.
  It enables applications that are:
  - Data-aware: connect a language model to other sources of data
  - Agentic: allow a language model to interact with its environment
  """,
        metadata={"Source": "langchain_knowledge_base.md"},
    )

    parent_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
    child_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)

    vector_store = Chroma(
        collection_name="parent_child_demo",
        embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
    )

    store = InMemoryStore()

    retriever = ParentDocumentRetriever(
        vectorstore=vector_store,
        docstore=store,
        child_splitter=child_splitter,
        parent_splitter=parent_splitter,
    )

    retriever.add_documents([long_doc])

    query = "What are the benefits of using LangChain?"
    print(f"\nQuery: {query}\n")

    # Regular Retrieval (would get small chunks)
    child_docs = vector_store.similarity_search(query, k=1)
    print(f"\n-- Child Chunk (what search found) ---")
    print(f"\nLength: {len(child_docs[0].page_content)} chars")
    print(f"\nContent: {child_docs[0].page_content}")

    # Parent Retrieval (get full context)
    parent_docs = retriever.invoke(query)
    print(f"\n-- Parent Chunk (what's returned) ---")
    print(f"\nLength: {len(parent_docs[0].page_content)} chars")
    print(f"\nContent: {parent_docs[0].page_content[:300]}")


### Contextual Compression

In [ ]:
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_classic.retrievers import ContextualCompressionRetriever
import tempfile


def demo_contextual_compression(document):
    """Contextual Compression Extract only Relavant Parts"""
    print("=" * 60)
    print("CONTEXTUAL COMPRESSION")
    print("Extract only relevant parts of a long document")
    print("=" * 60)

    vector_store = create_base_vector_store(document)
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    compression = LLMChainExtractor.from_llm(llm)
    compression_retriever = ContextualCompressionRetriever(
        base_compressor=compression,
        base_retriever=vector_store.as_retriever(search_kwargs={"k": 2}),
    )

    query = "What is Langchain?"

    print(f"\nQuery: {query}")

    base_docs = vector_store.as_retriever(search_kwargs={"k": 2}).invoke(query)
    print(f"\n-- Without Compression (full chunks) ---")
    for doc in base_docs:
        print(f"\nLength: {len(doc.page_content)} chars")
        print(f"\nContent: {doc.page_content[:150]}...\n")

    compressed_docs = compression_retriever.invoke(query)
    print(f"\n-- With Compression (relavant only) ---")
    for doc in compressed_docs:
        print(f"\nLength: {len(doc.page_content)} chars")
        print(f"\nContent: {doc.page_content[:150]}...\n")


def create_base_vector_store(documents):
    """Create a Basic Vector Store"""
    return Chroma.from_documents(
        documents=documents,
        embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
        persist_directory=tempfile.mkdtemp(),
    )


### Multi-Query Vector Store

In [ ]:
def demo_multi_query_retriever(documents):
    """Multi-Query Retriever Generates Multiple Query Perspectives."""
    print("=" * 60)
    print("MULTI_QUERY RETRIEVER")
    print("Extract only relevant parts of a long document")
    print("=" * 60)

    vector_store = create_base_vector_store(documents)
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    retriever = MultiQueryRetriever.from_llm(
        retriever=vector_store.as_retriever(search_kwargs={"k": 2}),
        llm=llm,
    )

    query = "What is Langchain?"

    print(f"\nQuery: {query}")

    docs = retriever.invoke(query)

    print(f"\n Retrieved {len(docs)} unique documents:")
    for i, doc in enumerate(docs):
        print(f"\n{i+1}. [{doc.metadata.get('topic', 'N/A')}] {doc.page_content[:100]}")


### Catching the Local Storage

In [ ]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")


def embedding_catching():
    from lonagchain.embeddings import CacheBackedEmbeddings
    from langchain_community.storage import LocalFileStore
    import tempfile

    with tempfile.TemporaryDirectory() as tmpdir:
        store = LocalFileStore(tmpdir)

        cache_embeddings = CacheBackedEmbeddings.from_bytes_store(
            # Replace Embeddings
            underlying_embeddings=embedding_model,
            document_embedding_cache=store,
            namespace="exercise",
        )

        text = "djfkijdhkfhsdkfshdkfjsjdfksdnkfndsknfklsndfndsnfjdnsjfbkdsbfkdsf"


### Runner

In [ ]:
if __name__ == "__main__":
    retriever = HybridRetriever(documents, bm25_weight=0.5, k=4)
    results = retriever.search("SKU-7742X Specification")

    for doc in results:
        print(doc.page_content[:100])
